# STAGE 1: Feature Engineering with Logistic Regression

**Goal**: Find the best feature set by testing incrementally with Logistic Regression

**Strategy**:
- Level 0: Basic encoding (baseline) ✅
- Level 1: + Standardization & OneHot encoding
- Level 2: + Derived features
- Level 3: + Interaction terms (optional)
- Level 4: Feature selection

**Output**: Best feature set to use in Stage 2

## Setup and Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

## Load Data

In [2]:
# Load raw training data
train_df = pd.read_csv('../data/train_data.csv')
print(f"Training data shape: {train_df.shape}")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nTarget distribution:")
print(train_df['Revenue'].value_counts())

# Separate features and target
X_raw = train_df.drop('Revenue', axis=1)
y = train_df['Revenue'].astype(int)

print(f"\nFeature dtypes:")
print(X_raw.dtypes)

Training data shape: (9864, 18)

Columns: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']

Target distribution:
Revenue
False    8338
True     1526
Name: count, dtype: int64

Feature dtypes:
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                       object
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                 object
Weeke

## Define Feature Groups

In [3]:
# Numeric features
numeric_features = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'
]

# Categorical features (will use OneHot encoding)
categorical_features = ['Month', 'VisitorType']

# Boolean features
boolean_features = ['Weekend']

# Numeric that might be treated as categorical (high cardinality)
id_like_features = ['OperatingSystems', 'Browser', 'Region', 'TrafficType']

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Boolean features ({len(boolean_features)}): {boolean_features}")
print(f"ID-like features ({len(id_like_features)}): {id_like_features}")

Numeric features (10): ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']
Categorical features (2): ['Month', 'VisitorType']
Boolean features (1): ['Weekend']
ID-like features (4): ['OperatingSystems', 'Browser', 'Region', 'TrafficType']


## Setup Cross-Validation

In [4]:
# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_pipeline(pipeline, X, y, cv, level_name):
    """Evaluate a preprocessing pipeline with Logistic Regression"""
    print(f"\n{'='*60}")
    print(f"Evaluating: {level_name}")
    print(f"{'='*60}")
    
    # Cross-validation scores
    f1_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='f1')
    roc_auc_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='roc_auc')
    precision_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='precision')
    recall_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='recall')
    
    print(f"\n5-Fold Cross-Validation Results:")
    print(f"  F1-Score:  {f1_scores.mean():.4f} (± {f1_scores.std():.4f})")
    print(f"  ROC-AUC:   {roc_auc_scores.mean():.4f} (± {roc_auc_scores.std():.4f})")
    print(f"  Precision: {precision_scores.mean():.4f} (± {precision_scores.std():.4f})")
    print(f"  Recall:    {recall_scores.mean():.4f} (± {recall_scores.std():.4f})")
    
    return {
        'level': level_name,
        'f1_mean': f1_scores.mean(),
        'f1_std': f1_scores.std(),
        'roc_auc_mean': roc_auc_scores.mean(),
        'roc_auc_std': roc_auc_scores.std(),
        'precision_mean': precision_scores.mean(),
        'recall_mean': recall_scores.mean()
    }

# Store results
results = []

## Level 0: Basic Encoding (Baseline)

Use the current preprocessing from `preprocess.py`:
- LabelEncoder for categoricals
- Boolean to int

In [5]:
# Load preprocessed data (Level 0)
train_df_level0 = pd.read_csv('../data/train_data_preprocessed.csv')
X_level0 = train_df_level0.drop('Revenue', axis=1)
y_level0 = train_df_level0['Revenue']

# Simple pipeline with just Logistic Regression
pipeline_level0 = Pipeline([
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Evaluate
result_level0 = evaluate_pipeline(pipeline_level0, X_level0, y_level0, cv, "Level 0: Basic Encoding")
results.append(result_level0)


Evaluating: Level 0: Basic Encoding

5-Fold Cross-Validation Results:
  F1-Score:  0.4869 (± 0.0360)
  ROC-AUC:   0.8726 (± 0.0093)
  Precision: 0.7291 (± 0.0276)
  Recall:    0.3663 (± 0.0369)


## Level 1: Add Standardization & OneHot Encoding

- StandardScaler for numeric features
- OneHotEncoder for categorical features
- Keep boolean as-is (0/1)

In [6]:
# Convert boolean to int first
X_level1 = X_raw.copy()
X_level1['Weekend'] = X_level1['Weekend'].astype(int)

# Combine all numeric features (including id-like and boolean)
all_numeric = numeric_features + id_like_features + boolean_features

# Create preprocessing pipeline
preprocessor_level1 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), all_numeric),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

pipeline_level1 = Pipeline([
    ('preprocessor', preprocessor_level1),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Evaluate
result_level1 = evaluate_pipeline(pipeline_level1, X_level1, y, cv, "Level 1: + Standardization & OneHot")
results.append(result_level1)


Evaluating: Level 1: + Standardization & OneHot

5-Fold Cross-Validation Results:
  F1-Score:  0.4997 (± 0.0218)
  ROC-AUC:   0.8933 (± 0.0059)
  Precision: 0.7455 (± 0.0223)
  Recall:    0.3761 (± 0.0218)


## Level 2: Add Derived Features

Add behavioral features:
- avg_time_per_page
- total_duration
- bounce_exit_ratio
- interaction_intensity

In [7]:
def add_derived_features(X):
    """Add derived features"""
    X = X.copy()
    
    # Average time per page
    total_pages = X['ProductRelated'] + X['Informational'] + X['Administrative'] + 1e-5
    total_time = X['ProductRelated_Duration'] + X['Informational_Duration'] + X['Administrative_Duration']
    X['avg_time_per_page'] = total_time / total_pages
    
    # Total session duration
    X['total_duration'] = total_time
    
    # Bounce to exit ratio
    X['bounce_exit_ratio'] = X['BounceRates'] / (X['ExitRates'] + 1e-5)
    
    # Interaction intensity (total pages visited)
    X['interaction_intensity'] = X['ProductRelated'] + X['Informational'] + X['Administrative']
    
    return X

# Create feature creator
feature_creator = FunctionTransformer(add_derived_features)

# Update numeric features list
derived_numeric_features = all_numeric + [
    'avg_time_per_page', 'total_duration', 'bounce_exit_ratio', 'interaction_intensity'
]

# Create preprocessing pipeline
preprocessor_level2 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), derived_numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

pipeline_level2 = Pipeline([
    ('feature_creator', feature_creator),
    ('preprocessor', preprocessor_level2),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Evaluate
result_level2 = evaluate_pipeline(pipeline_level2, X_level1, y, cv, "Level 2: + Derived Features")
results.append(result_level2)


Evaluating: Level 2: + Derived Features

5-Fold Cross-Validation Results:
  F1-Score:  0.5002 (± 0.0195)
  ROC-AUC:   0.8925 (± 0.0061)
  Precision: 0.7476 (± 0.0214)
  Recall:    0.3761 (± 0.0197)


## Level 3: Add Interaction Terms (Optional)

Add interaction features:
- VisitorType × Month
- VisitorType × Weekend

In [8]:
# Create interaction features manually
def add_interaction_features(X):
    """Add interaction features"""
    X = X.copy()
    
    # VisitorType × Weekend
    X['visitor_weekend'] = X['VisitorType'].astype(str) + '_' + X['Weekend'].astype(str)
    
    # VisitorType × Month
    X['visitor_month'] = X['VisitorType'].astype(str) + '_' + X['Month'].astype(str)
    
    return X

interaction_creator = FunctionTransformer(add_interaction_features)

# Update categorical features
interaction_categorical_features = categorical_features + ['visitor_weekend', 'visitor_month']

preprocessor_level3 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), derived_numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), interaction_categorical_features)
    ]
)

pipeline_level3 = Pipeline([
    ('interaction_creator', interaction_creator),
    ('feature_creator', feature_creator),
    ('preprocessor', preprocessor_level3),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Evaluate
result_level3 = evaluate_pipeline(pipeline_level3, X_level1, y, cv, "Level 3: + Interaction Terms")
results.append(result_level3)


Evaluating: Level 3: + Interaction Terms

5-Fold Cross-Validation Results:
  F1-Score:  0.5047 (± 0.0208)
  ROC-AUC:   0.8911 (± 0.0065)
  Precision: 0.7472 (± 0.0214)
  Recall:    0.3814 (± 0.0210)


## Level 4: Feature Selection

Test different feature selection methods:
1. Correlation-based
2. L1 regularization (Lasso)
3. Top features from tree model

In [9]:
# Method 1: L1 Regularization (Lasso)
pipeline_level4_l1 = Pipeline([
    ('feature_creator', feature_creator),
    ('preprocessor', preprocessor_level2),
    ('classifier', LogisticRegression(
        penalty='l1',
        C=0.1,  # Strong regularization
        solver='saga',
        max_iter=1000,
        random_state=42
    ))
])

result_level4_l1 = evaluate_pipeline(pipeline_level4_l1, X_level1, y, cv, "Level 4: L1 Feature Selection")
results.append(result_level4_l1)


Evaluating: Level 4: L1 Feature Selection

5-Fold Cross-Validation Results:
  F1-Score:  0.4974 (± 0.0231)
  ROC-AUC:   0.8938 (± 0.0051)
  Precision: 0.7519 (± 0.0233)
  Recall:    0.3722 (± 0.0246)


## Compare All Levels

In [10]:
# Create comparison table
results_df = pd.DataFrame(results)
results_df = results_df.round(4)

print("\n" + "="*80)
print("FEATURE ENGINEERING COMPARISON")
print("="*80)
print(results_df.to_string(index=False))

# Find best level
best_idx = results_df['f1_mean'].idxmax()
best_level = results_df.loc[best_idx]

print("\n" + "="*80)
print("BEST FEATURE LEVEL")
print("="*80)
print(f"Level: {best_level['level']}")
print(f"F1-Score: {best_level['f1_mean']:.4f} (± {best_level['f1_std']:.4f})")
print(f"ROC-AUC: {best_level['roc_auc_mean']:.4f} (± {best_level['roc_auc_std']:.4f})")


FEATURE ENGINEERING COMPARISON
                              level  f1_mean  f1_std  roc_auc_mean  roc_auc_std  precision_mean  recall_mean
            Level 0: Basic Encoding   0.4869  0.0360        0.8726       0.0093          0.7291       0.3663
Level 1: + Standardization & OneHot   0.4997  0.0218        0.8933       0.0059          0.7455       0.3761
        Level 2: + Derived Features   0.5002  0.0195        0.8925       0.0061          0.7476       0.3761
       Level 3: + Interaction Terms   0.5047  0.0208        0.8911       0.0065          0.7472       0.3814
      Level 4: L1 Feature Selection   0.4974  0.0231        0.8938       0.0051          0.7519       0.3722

BEST FEATURE LEVEL
Level: Level 3: + Interaction Terms
F1-Score: 0.5047 (± 0.0208)
ROC-AUC: 0.8911 (± 0.0065)


## Save Best Feature Set

Save the preprocessing pipeline for use in Stage 2

In [ ]:
import joblib

# Train best pipeline on full training set
# (Replace with the actual best pipeline from above)
best_pipeline = pipeline_level2  # Example: Level 2 was best
best_pipeline.fit(X_level1, y)

# Save pipeline
joblib.dump(best_pipeline, '../models/best_preprocessing_pipeline.pkl')
print("Best preprocessing pipeline saved to: models/best_preprocessing_pipeline.pkl")

# Also save preprocessed data for Stage 2
X_transformed = best_pipeline.named_steps['preprocessor'].fit_transform(
    best_pipeline.named_steps['feature_creator'].transform(X_level1)
)

# Get feature names
feature_names = best_pipeline.named_steps['preprocessor'].get_feature_names_out()
X_best = pd.DataFrame(X_transformed, columns=feature_names)
X_best['Revenue'] = y.values

# Save
X_best.to_csv('../data/train_data_best_features.csv', index=False)
print("Best feature set saved to: data/train_data_best_features.csv")
print(f"Number of features: {len(feature_names)}")

## Summary

**Output for Stage 2:**
- Best preprocessing pipeline saved
- Best feature set saved as CSV
- Ready to test all models in Stage 2!